# 📊 Data Jobs Market Tracker — Dashboard

Análisis del mercado global de empleos en datos (Data Analyst, Data Engineer, Data Scientist, BI, ML Engineer)
recolectado vía Adzuna API en 7 países: US, GB, DE, BR, ES, NL, IT.

**Pipeline completo:** Adzuna API → pandas cleaning → PostgreSQL → este dashboard.

| Etapa | Archivo |
|---|---|
| Extracción | `adzuna_api.py` |
| Limpieza | `clean.py` |
| Carga a DB | `load_to_db.py` |
| Análisis SQL | `analysis.sql` |
| Visualización | `dashboard.ipynb` (este notebook) |


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# IMPORTANT: nbconvert cannot represent the native Plotly mimetype
# (application/vnd.plotly.v1+json) produced by fig.show(). Instead of
# relying on a Plotly renderer, we render each figure to static HTML/JS
# ourselves and display it as text/html, which nbconvert always supports.
_plotlyjs_included = {"done": False}

def show_fig(fig):
    """Display a Plotly figure as embedded HTML (works in Jupyter and in nbconvert exports)."""
    include_js = "cdn" if not _plotlyjs_included["done"] else False
    _plotlyjs_included["done"] = True
    display(HTML(fig.to_html(include_plotlyjs=include_js, full_html=False)))

load_dotenv()

DB_USER     = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME", "adventureworks")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

df = pd.read_sql("SELECT * FROM dw.data_jobs_market", engine)
print(f"Registros cargados: {len(df):,}")
df.head()

## 1. Volumen de empleos por país

In [ ]:
country_summary = (
    df.groupby("country")
      .agg(total_jobs=("job_id", "count"),
           remote_jobs=("is_remote", "sum"),
           with_salary=("salary_avg", lambda x: x.notna().sum()))
      .sort_values("total_jobs", ascending=False)
      .reset_index()
)

fig = px.bar(
    country_summary,
    x="country", y="total_jobs",
    color="total_jobs",
    color_continuous_scale="Blues",
    text="total_jobs",
    title="Volumen total de ofertas de datos por país",
    labels={"country": "País", "total_jobs": "Total de ofertas"}
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, height=450)
show_fig(fig)

## 2. Roles más demandados

In [ ]:
role_demand = (
    df.groupby("search_term")
      .size()
      .sort_values(ascending=False)
      .reset_index(name="postings")
)

fig = px.bar(
    role_demand,
    x="postings", y="search_term",
    orientation="h",
    color="postings",
    color_continuous_scale="Teal",
    title="Demanda total por rol (todos los países)",
    labels={"search_term": "Rol", "postings": "Cantidad de ofertas"}
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450, showlegend=False)
show_fig(fig)

## 3. Distribución salarial por seniority

In [ ]:
salary_df = df[
    (df["salary_avg"].notna()) &
    (df["salary_avg"] > 10000) &
    (df["seniority"] != "Not specified")
]

fig = px.box(
    salary_df,
    x="seniority", y="salary_avg",
    color="seniority",
    category_orders={"seniority": ["Junior", "Mid", "Senior"]},
    title="Distribución salarial por nivel de seniority (USD)",
    labels={"seniority": "Seniority", "salary_avg": "Salario promedio (USD)"}
)
fig.update_layout(height=500, showlegend=False)
show_fig(fig)

## 4. Salario promedio por rol

In [ ]:
role_salary = (
    salary_df.groupby("search_term")["salary_avg"]
    .agg(["mean", "median", "count"])
    .query("count >= 5")
    .sort_values("median", ascending=False)
    .reset_index()
)

fig = px.bar(
    role_salary,
    x="median", y="search_term",
    orientation="h",
    color="median",
    color_continuous_scale="Sunset",
    text=role_salary["median"].round(0).astype(int).astype(str),
    title="Salario mediano por rol (USD) — solo roles con 5+ ofertas con salario",
    labels={"search_term": "Rol", "median": "Salario mediano (USD)"}
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450, showlegend=False)
show_fig(fig)

## 5. Mezcla de seniority por país

In [ ]:
seniority_country = (
    df[df["seniority"] != "Not specified"]
    .groupby(["country", "seniority"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    seniority_country,
    x="country", y="count", color="seniority",
    barmode="stack",
    category_orders={"seniority": ["Junior", "Mid", "Senior"]},
    color_discrete_map={"Junior": "#90CAF9", "Mid": "#42A5F5", "Senior": "#1565C0"},
    title="Distribución de seniority por país",
    labels={"country": "País", "count": "Cantidad de ofertas", "seniority": "Nivel"}
)
fig.update_layout(height=450)
show_fig(fig)

## 6. % de ofertas remotas por rol

In [ ]:
remote_by_role = (
    df.groupby("search_term")
      .agg(total=("job_id", "count"), remote=("is_remote", "sum"))
      .assign(remote_pct=lambda x: (x["remote"] / x["total"] * 100).round(1))
      .sort_values("remote_pct", ascending=False)
      .reset_index()
)

fig = px.bar(
    remote_by_role,
    x="remote_pct", y="search_term",
    orientation="h",
    color="remote_pct",
    color_continuous_scale="Greens",
    text=remote_by_role["remote_pct"].astype(str) + "%",
    title="Porcentaje de ofertas remotas por rol",
    labels={"search_term": "Rol", "remote_pct": "% remoto"}
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450, showlegend=False)
show_fig(fig)

---

### 🔑 Insights clave

- **US y GB** concentran el mayor volumen de ofertas y son los mercados con mejor reporte de salarios.
- El salto salarial **Junior → Senior** es de aproximadamente 2x en la mediana.
- **Data Engineer** es el rol con mayor demanda absoluta; **Data Scientist** y **ETL Developer** ofrecen los salarios medianos más altos.
- **Alemania (DE)** tiene alto volumen de ofertas pero baja transparencia salarial (pocas publican el salario).
- Los Países Bajos (NL) muestran la menor proporción de trabajo remoto, sugiriendo una cultura laboral más presencial.

---
*Proyecto desarrollado por Juan Bautista Acuña — [GitHub](https://github.com/bautistaacuna/DataAnalysis) | [Tableau Public](https://public.tableau.com/app/profile/juan.bautista.acuna)*
